In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


RAW_DATA_PATH = Path().resolve().parent.parent / "data" / "raw" / "resultados"
OUTPUT_DATA_PATH = Path().resolve().parent.parent / "data" / "processed"


def processar_resultados_eleitorais(raw_path):

    cols = [
        "ANO_ELEICAO",
        "NR_TURNO",
        "SG_UF",
        "CD_MUNICIPIO",
        "NM_MUNICIPIO",
        "DS_CARGO",
        "NR_CANDIDATO",
        "NM_CANDIDATO",
        "SG_PARTIDO",
        "DS_COMPOSICAO_COLIGACAO",
        "DS_SIT_TOT_TURNO",
        "QT_VOTOS_NOMINAIS",
        "NM_TIPO_ELEICAO"
    ]

    dfs_resultados = {}

    for ano in range(1998, 2024, 2):

        dfs_resultados[str(ano)] = pd.read_csv(
            raw_path / f"votacao_candidato_munzona_{ano}_BRASIL.csv",
            sep=";",
            encoding="latin1",
            usecols=cols,
            dtype={
                "ANO_ELEICAO": int,
                "NR_TURNO": int,
                "SG_UF": str,
                "CD_MUNICIPIO": str,
                "NM_MUNICIPIO": str,
                "DS_CARGO": str,
                "NR_CANDIDATO": str,
                "NM_CANDIDATO": str,
                "SG_PARTIDO": str,
                "DS_COMPOSICAO_COLIGACAO": str,
                "DS_SIT_TOT_TURNO": str,
                "QT_VOTOS_NOMINAIS": int,
                "NM_TIPO_ELEICAO": str
            },
        )

        dfs_resultados[str(ano)].columns = dfs_resultados[str(ano)].columns.str.lower()


        dfs_resultados[str(ano)]['nm_tipo_eleicao'] = dfs_resultados[str(ano)]['nm_tipo_eleicao'].str.upper()
        dfs_resultados[str(ano)] = dfs_resultados[str(ano)].loc[dfs_resultados[str(ano)]['nm_tipo_eleicao'] == "ELEIÇÃO ORDINÁRIA"]

        dfs_resultados[str(ano)] = (
            dfs_resultados[str(ano)]
            .groupby(
                [
                    "ano_eleicao",
                    "nr_turno",
                    "sg_uf",
                    "cd_municipio",
                    "nm_municipio",
                    "ds_cargo",
                    "nr_candidato",
                    "nm_candidato",
                    "sg_partido",
                    "ds_composicao_coligacao",
                    "ds_sit_tot_turno",
                ],
                as_index=False,
            )
            .agg({"qt_votos_nominais": "sum"})
        )

    return pd.concat(dfs_resultados, axis=0).reset_index(drop=True)


def retirar_resultados_municipais_cargos_estaduais(resultados):

    cargos_estaduais = [
        "Deputado Estadual",
        "Deputado Federal",
        "Governador",
        "Senador",
        "Deputado Distrital",
    ]
    cargos_municipais = ["Prefeito", "Vereador"]

    resultados_munic = resultados.loc[
        resultados["ds_cargo"].isin(cargos_municipais)
    ].copy()

    resultados_estd = (
        resultados.loc[resultados["ds_cargo"].isin(cargos_estaduais)]
        .groupby(
            [
                "ano_eleicao",
                "nr_turno",
                "sg_uf",
                "ds_cargo",
                "nr_candidato",
                "nm_candidato",
                "sg_partido",
                "ds_composicao_coligacao",
                "ds_sit_tot_turno",
            ],
            as_index=False,
        )
        .agg({"qt_votos_nominais": "sum"})
        .assign(cd_municipio=np.nan, nm_municipio=np.nan)
    )

    return pd.concat([resultados_munic, resultados_estd], axis=0)


def definir_turno_eleicao(resultados_tratado):
    resultados_tratado[["ano_eleicao", "sg_uf", "cd_municipio", "ds_cargo"]] = (
        resultados_tratado[["ano_eleicao", "sg_uf", "cd_municipio", "ds_cargo"]].fillna(
            "NA"
        )
    )

    turnos = (
        resultados_tratado.groupby(
            ["ano_eleicao", "sg_uf", "cd_municipio", "ds_cargo"], as_index=False
        )
        .agg({"nr_turno": "unique"})
        .astype({"nr_turno": str})
        .rename(columns={"nr_turno": "turnos_eleicao"})
    )

    resultados_tratado_definicao_turnos = pd.merge(
        resultados_tratado,
        turnos,
        on=["ano_eleicao", "sg_uf", "cd_municipio", "ds_cargo"],
        how="left",
    )

    resultados_tratado_turnos = resultados_tratado_definicao_turnos.loc[
        np.where(
            resultados_tratado_definicao_turnos["turnos_eleicao"] == "[1 2]",
            resultados_tratado_definicao_turnos["nr_turno"] == 2,
            resultados_tratado_definicao_turnos["nr_turno"] == 1,
        )
    ]
    return resultados_tratado_turnos


def consolidar_salvar(raw_path, output_path):

    resultados = processar_resultados_eleitorais(raw_path)
    resultados_tratado = retirar_resultados_municipais_cargos_estaduais(resultados)
    resultados_tratado = definir_turno_eleicao(resultados_tratado)

    resultados_tratado.to_parquet(
        output_path / "resultados.parquet", index=False, engine="pyarrow"
    )
    print(f"Arquivo salvo com sucesso em: {output_path}")
    return resultados


resultados = consolidar_salvar(RAW_DATA_PATH, OUTPUT_DATA_PATH)

Arquivo salvo com sucesso em: C:\Users\yuri.taba\OneDrive - Eicon Controles Inteligentes de Negocios Ltda\0_Documentos\dcp\gastos-campanha\data\processed
